In [1]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import requests
from PIL import Image

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

2026-02-22 11:23:14.356140: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771759394.559236      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771759394.615897      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771759395.069290      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771759395.069335      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771759395.069338      24 computation_placer.cc:177] computation placer alr

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-handwritten and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
image = Image.open("/kaggle/input/datasets/timwalker679/test124/Screenshot 2026-02-19 203708.png").convert("RGB")

pixel_values = processor(image, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values)

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(generated_text)

t. Did show .


In [3]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

class PrescriptionDataset(Dataset):
    def __init__(self, csv_file, image_dir, processor, max_length=16):
        self.df = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = os.path.join(self.image_dir, row["IMAGE"])
        image = Image.open(image_path).convert("RGB")

        pixel_values = self.processor(
            image,
            return_tensors="pt"
        ).pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            row["MEDICINE_NAME"],
            padding="max_length",
            max_length=self.max_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)
        
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

import torch

class TrOCRDataCollator:
    def __call__(self, features):
        pixel_values = torch.stack([f["pixel_values"] for f in features])
        labels = torch.stack([f["labels"] for f in features])

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }



In [4]:
ROOT = "/kaggle/input/datasets/mamun1113/doctors-handwritten-prescription-bd-dataset/Doctor’s Handwritten Prescription BD dataset"

train_dataset = PrescriptionDataset(
    csv_file=f"{ROOT}/Training/training_labels.csv",
    image_dir=f"{ROOT}/Training/training_words",
    processor=processor
)

val_dataset = PrescriptionDataset(
    csv_file=f"{ROOT}/Validation/validation_labels.csv",
    image_dir=f"{ROOT}/Validation/validation_words",
    processor=processor
)

In [5]:
sample = train_dataset[0]

print(sample["pixel_values"].shape)
print(sample["labels"][:10])


torch.Size([3, 384, 384])
tensor([    0, 26145,  8152,     2,  -100,  -100,  -100,  -100,  -100,  -100])


In [6]:
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

data_collator = TrOCRDataCollator()
args = Seq2SeqTrainingArguments(
    output_dir="./trocr-prescription",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-6,
    num_train_epochs=12,
    warmup_ratio=0.1,
    fp16=True,
    save_strategy="no",
    
    # evaluation_strategy="steps",
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=processor.tokenizer
)

trainer.train()



/tmp/ipykernel_24/3021881563.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,1.781200
1000,0.044900
1500,0.008000
2000,0.003300


TrainOutput(global_step=2340, training_loss=0.39297213854952756, metrics={'train_runtime': 3980.3692, 'train_samples_per_second': 9.406, 'train_steps_per_second': 0.588, 'total_flos': 2.801579755088904e+19, 'train_loss': 0.39297213854952756, 'epoch': 12.0})

In [7]:
# val_metrics = trainer.evaluate(val_dataset)
# print(val_metrics)
trainer.save_model("./trocr-prescription/final")
processor.save_pretrained("./trocr-prescription/final")

[]

In [8]:
!pip install jiwer
import jiwer
import torch
import jiwer

def evaluate_ocr(model, processor, dataset):
    preds = []
    gts = []

    model.eval()
    for sample in dataset:
        pixel_values = sample["pixel_values"].unsqueeze(0).to(model.device)

        with torch.no_grad():
            ids = model.generate(pixel_values,min_length=1,num_beams=5,length_penalty=1.2,early_stopping=True)

        pred = processor.batch_decode(
            ids, skip_special_tokens=True
        )[0].strip()

        labels = sample["labels"].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id

        gt = processor.tokenizer.decode(
            labels,
            skip_special_tokens=True
        ).strip()

        preds.append(pred)
        gts.append(gt)

    cer = jiwer.cer(gts, preds)
    return cer

print(evaluate_ocr(model, processor, val_dataset))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.4 MB/s eta 0:00:00
0.09412955465587045


In [9]:
for i in range(10):
    sample = val_dataset[10*i]
    pixel_values = sample["pixel_values"].unsqueeze(0).to(model.device)

    ids = model.generate(pixel_values)

    pred = processor.batch_decode(ids, skip_special_tokens=True)[0]

    labels = sample["labels"].clone()
    labels[labels == -100] = processor.tokenizer.pad_token_id
    gt = processor.tokenizer.decode(labels, skip_special_tokens=True)

    print(f"GT   : '{gt}'")
    print(f"PRED : '{pred}'")
    print("-"*40)


GT   : 'Aceta'
PRED : 'Aceta'
----------------------------------------
GT   : 'Ace'
PRED : 'Ace'
----------------------------------------
GT   : 'Alatrol'
PRED : 'Alcatrol'
----------------------------------------
GT   : 'Amodis'
PRED : 'Amodis'
----------------------------------------
GT   : 'Atrizin'
PRED : 'Atrizin'
----------------------------------------
GT   : 'Axodin'
PRED : 'Axodin'
----------------------------------------
GT   : 'Azithrocin'
PRED : 'Azithrocin'
----------------------------------------
GT   : 'Azyth'
PRED : 'Azyth'
----------------------------------------
GT   : 'Az'
PRED : 'Az'
----------------------------------------
GT   : 'Bacaid'
PRED : 'Becaid'
----------------------------------------


In [10]:
image = Image.open("/kaggle/input/datasets/timwalker679/test124/Screenshot 2026-02-19 203708.png").convert("RGB")
# pixel_values = image.unsqueeze(0).to(model.device)
pixel_values = processor(image, return_tensors="pt").pixel_values.to(model.device)

# pixel_values = processor(image,return_tensors="pt").pixel_values.squeeze(0)
generated_ids = model.generate(pixel_values)

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(generated_text)

Dilnov


In [11]:
def word_accuracy(model, processor, dataset):
    correct = 0
    total = len(dataset)

    model.eval()
    with torch.no_grad():
        for sample in dataset:
            pixel_values = sample["pixel_values"].unsqueeze(0).to(model.device)

            ids = model.generate(pixel_values, max_length=16,min_length=1,num_beams=5,length_penalty=1.2,early_stopping=True)

            pred = processor.batch_decode(
                ids, skip_special_tokens=True
            )[0].strip().lower()

            labels = sample["labels"].clone()
            labels[labels == -100] = processor.tokenizer.pad_token_id
            gt = processor.tokenizer.decode(
                labels, skip_special_tokens=True
            ).strip().lower()

            if pred == gt:
                correct += 1

    return correct / total
acc = word_accuracy(model, processor, val_dataset)
print("Word Accuracy:", acc)

Word Accuracy: 0.7384615384615385


In [12]:
# from transformers import TrOCRProcessor, VisionEncoderDecoderModel
# import torch

# model_path = "/kaggle/working/trocr-prescription/final"

# processor = TrOCRProcessor.from_pretrained(model_path)
# model = VisionEncoderDecoderModel.from_pretrained(model_path)

# device = "cuda" if torch.cuda.is_available() else "cpu"
# model.to(device)
# model.eval()
# from PIL import Image

# image = Image.open("/kaggle/input/datasets/timwalker679/testing/do.png").convert("RGB")

# pixel_values = processor(images=image, return_tensors="pt").pixel_values
# pixel_values = pixel_values.to(device)

# with torch.no_grad():
#     generated_ids = model.generate(pixel_values)

# text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
# print(text)
